<a href="https://colab.research.google.com/github/Ranjith0805/AI-Project/blob/main/AI_Project.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install google-generativeai streamlit pyngrok -q

In [ ]:
import google.generativeai as genai
from google.colab import userdata

# Get API key from secrets
api_key = userdata.get('GEMINI_API_KEY')

# Connect to Gemini
genai.configure(api_key=api_key)

# Choose model
model = genai.GenerativeModel('gemini-2.5-flash')

print("Gemini connected successfully!")

In [7]:
%%writefile app.py

import streamlit as st #3
import google.generativeai as genai #4
import os #5
import json #6

# Configure Gemini #8
genai.configure(api_key=os.environ.get('GEMINI_API_KEY')) #9

# Define model #11
model = genai.GenerativeModel('gemini-2.5-flash') #12

# Sidebar #14
with st.sidebar: #15
    st.title("📚 About") #16
    st.write("Welcome to Siri's Study Buddy!") #17
    st.write("Ask me anything and I'll explain it simply!") #18
    st.divider() #19
    subject = st.selectbox( #20
        "📖 Choose Subject", #21
        ["General", "Mathematics", "Physics", "Chemistry", "Biology", "History", "Computer Science"] #22
    ) #23
    difficulty = st.selectbox( #24
        "🎯 Choose Difficulty", #25
        ["Easy", "Medium", "Hard"] #26
    ) #27
    language = st.selectbox( #28
        "🌍 Choose Language", #29
        ["English", "Tamil", "Hindi", "Telugu", "Kannada"] #30
    ) #31
    st.divider() #32
    if st.button("🎯 Generate Quiz"): #33
        recent_chat = "" #34
        if len(st.session_state.chat_history) > 2: #35
            for msg in st.session_state.chat_history[-4:]: #36
                if msg["role"] == "user": #37
                    recent_chat += msg["parts"][0] + " " #38
        quiz_prompt = f"""Generate 5 {difficulty} level multiple choice questions on {subject}. #39
{'Focus on these topics we just discussed: ' + recent_chat if recent_chat else ''} #40
Return ONLY a JSON array like this, nothing else: #41
[ #42
  {{ #43
    "question": "question here", #44
    "options": ["A. option1", "B. option2", "C. option3", "D. option4"], #45
    "answer": "A. option1" #46
  }} #47
]""" #48
        response = model.generate_content(quiz_prompt) #49
        raw = response.text.strip() #50
        raw = raw.replace("```json", "").replace("```", "").strip() #51
        try: #52
            st.session_state.quiz = json.loads(raw) #53
            st.session_state.quiz_answers = {} #54
            st.session_state.quiz_submitted = False #55
        except: #56
            st.error("Quiz generation failed! Try again.") #57
        st.rerun() #58
    st.divider() #59
    st.subheader("📝 Summarizer") #60
    text_to_summarize = st.text_area("Paste text here to summarize:") #61
    if st.button("✨ Summarize!"): #62
        if text_to_summarize: #63
            summary_prompt = f"Summarize this text in simple points:\n\n{text_to_summarize}" #64
            st.session_state.chat_history.append({ #65
                "role": "user", #66
                "parts": [summary_prompt] #67
            }) #68
            response = model.generate_content(st.session_state.chat_history) #69
            ai_reply = response.text #70
            st.session_state.chat_history.append({ #71
                "role": "model", #72
                "parts": [ai_reply] #73
            }) #74
            st.rerun() #75
    st.divider() #76
    if st.button("🗑️ Clear Chat"): #77
        st.session_state.chat_history = [ #78
            { #79
                "role": "user", #80
                "parts": [f"You are a friendly AI Study Buddy specializing in {subject}. Always respond in {language}. Difficulty level is {difficulty} — if Easy explain like a 10 year old with very simple words, if Medium explain normally with examples, if Hard give deep technical explanation. At the end always ask if they understood."] #81
            }, #82
            { #83
                "role": "model", #84
                "parts": ["Got it! I'm your Study Buddy! What would you like to learn today? 😊"] #85
            } #86
        ] #87
        st.rerun() #88
# About section
with st.expander("ℹ️ About this app"):
    st.write("""
    ## 🤖 Siri's Study Buddy

    An AI powered personal study assistant built with Python and Gemini AI.
    Ask questions, generate quizzes, summarize text — all in your language!

    ---

    ### 🎯 What Problem Does It Solve?
    Students often struggle to:
    - Understand complex topics simply
    - Generate practice questions
    - Summarize large texts

    Siri's Study Buddy solves all three in one simple app!

    ---

    ### ✅ Features:
    - 💬 **AI Chat** — Ask anything, get simple explanations with examples
    - 📖 **Subject Selector** — Mathematics, Physics, Chemistry, Biology, History, Computer Science
    - 🎯 **Difficulty Selector** — Easy, Medium, Hard
    - 🌍 **Language Selector** — English, Tamil, Hindi, Telugu, Kannada
    - 🎯 **Interactive Quiz** — Generate 5 MCQ questions with scoring
    - 📝 **Text Summarizer** — Paste any text and get bullet point summary
    - 🗑️ **Clear Chat** — Reset conversation anytime

    ---

    ### 🛠️ Tech Stack:
    - **Python** — Core programming language
    - **Gemini AI** — AI brain of the app
    - **Streamlit** — Web UI framework
    - **Google Colab** — Cloud development
    - **GitHub** — Version control

    ---

    ### 👨‍💻 Developer:
    **Ranjith** —  Built a complete AI app step by step! 💪

    > *"Building technology that makes a difference!"* 🌱

    ---

    ### ⚠️ Note:
    This app uses Gemini AI free tier.
    If you see quota exceeded error, wait 1-2 hours and try again!
    """)#89
# App title #90
st.title("🤖 Siri's Study Buddy") #91
st.subheader("Ask me anything — I'll explain it simply!") #92

# Welcome message - shows only once #94
if "welcomed" not in st.session_state: #95
    st.session_state.welcomed = True #96
    st.success("👋 Welcome to Siri's Study Buddy! Select your subject, difficulty and language from the sidebar and start learning! 🚀") #97

# Initialize chat history #99
if "chat_history" not in st.session_state: #100
    st.session_state.chat_history = [ #101
        { #102
            "role": "user", #103
            "parts": [f"You are a friendly AI Study Buddy specializing in {subject}. Always respond in {language}. Difficulty level is {difficulty} — if Easy explain like a 10 year old with very simple words, if Medium explain normally with examples, if Hard give deep technical explanation. At the end always ask if they understood."] #104
        }, #105
        { #106
            "role": "model", #107
            "parts": ["Got it! I'm your Study Buddy! What would you like to learn today? 😊"] #108
        } #109
    ] #110

# Show quiz if generated #112
if "quiz" in st.session_state and st.session_state.quiz: #113
    st.subheader("📝 Quiz Time!") #114
    for i, q in enumerate(st.session_state.quiz): #115
        st.write(f"**Q{i+1}. {q['question']}**") #116
        st.session_state.quiz_answers[i] = st.radio( #117
            f"Select answer for Q{i+1}:", #118
            q["options"], #119
            key=f"q{i}" #120
        ) #121
    if st.button("✅ Submit Quiz"): #122
        score = 0 #123
        for i, q in enumerate(st.session_state.quiz): #124
            if st.session_state.quiz_answers.get(i) == q["answer"]: #125
                score += 1 #126
        st.success(f"🎉 Your Score: {score}/5!") #127
        if score == 5: #128
            st.balloons() #129
            st.write("Perfect score! Amazing!! 🌟") #130
        elif score >= 3: #131
            st.write("Good job! Keep it up! 💪") #132
        else: #133
            st.write("Keep practicing! You'll get there! 😊") #134
        st.session_state.quiz = [] #135

# Show chat messages #137
st.divider() #138
st.subheader("💬 Chat") #139
for message in st.session_state.chat_history[2:]: #140
    if message["role"] == "user": #141
        st.chat_message("user").write(message["parts"][0]) #142
    else: #143
        st.chat_message("assistant").write(message["parts"][0]) #144

# Input box #146
user_input = st.chat_input("Ask your question here...") #147

if user_input: #149
    st.session_state.chat_history.append({ #150
        "role": "user", #151
        "parts": [user_input] #152
    }) #153
    response = model.generate_content(st.session_state.chat_history) #154
    ai_reply = response.text #155
    st.session_state.chat_history.append({ #156
        "role": "model", #157
        "parts": [ai_reply] #158
    }) #159
    st.rerun() #160

Overwriting app.py


In [8]:
from google.colab import userdata
import os
import subprocess
from pyngrok import ngrok

# Kill any old tunnels
ngrok.kill()

# Set tokens from secrets
os.environ['GEMINI_API_KEY'] = userdata.get('GEMINI_API_KEY')
ngrok.set_auth_token(userdata.get('NGROK_TOKEN'))

# Start Streamlit
process = subprocess.Popen(
    ["streamlit", "run", "app.py", "--server.port", "8501"],
    stdout=subprocess.PIPE,
    stderr=subprocess.PIPE,
    env=os.environ
)

# Create public link
public_url = ngrok.connect(8501)
print(f"Your app is live at: {public_url}")

Your app is live at: NgrokTunnel: "https://casualty-fondness-barber.ngrok-free.dev" -> "http://localhost:8501"


In [ ]:
import google.generativeai as genai
from google.colab import userdata

genai.configure(api_key=userdata.get('GEMINI_API_KEY'))

for m in genai.list_models():
    if 'generateContent' in m.supported_generation_methods:
        print(m.name)